# Inferencia con `modelo_v2_mejor.keras`

Notebook para cargar el modelo entrenado y hacer predicciones sobre imágenes nuevas (NORMAL vs PNEUMONIA). Ajusta la ruta del modelo de ser necesario.
Puedes subir las imagenes corriendo el notebook y en un momento te pedira una imagen para probar el modelo.


## 1. Configuración
- Ajusta `MODEL_PATH` a la ubicación real del modelo guardado (`modelo_v2_mejor.keras`).
- Define `THRESHOLD` si quieres más sensibilidad (menor umbral) o más precisión (mayor umbral).


In [1]:
import os, numpy as np, tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

# Ruta al modelo entrenado (ajusta según dónde lo guardaste)
MODEL_PATH = r".\modelo_v2_mejor.keras"

# Configuración de entrada
IMG_SIZE = (180, 180)
THRESHOLD = 0.5
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

assert os.path.isfile(MODEL_PATH), f"No se encontró el modelo en: {MODEL_PATH}"
model = tf.keras.models.load_model(MODEL_PATH)
print("Modelo cargado:", MODEL_PATH)


Modelo cargado: .\modelo_v2_mejor.keras


## 2. Funciones de utilidad
- `load_image`: carga y normaliza la imagen.
- `predict_image`: devuelve probabilidad y clase.
- Usa `THRESHOLD` para decidir la clase (por defecto 0.5).


In [2]:
def load_image(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0), img

def predict_image(path, threshold=THRESHOLD):
    x, img = load_image(path)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob >= threshold else CLASS_NAMES[0]
    return prob, label, img


## 3. Inferencia sobre una imagen (con carga en notebook)
Usa el botón para subir una imagen (jpg/png). La celda mostrará la probabilidad y la imagen. Si prefieres usar una ruta fija, ajusta `image_path` abajo del todo.


In [3]:
import io, ipywidgets as widgets
from IPython.display import display

upload_single = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
out_single = widgets.Output()


def _extract_file(value):
    if isinstance(value, dict) and value:
        return next(iter(value.items()))
    if isinstance(value, (list, tuple)) and len(value) > 0:
        uf = value[0]
        return uf.name, {'content': uf.content}
    return None, None


def on_upload_single(change):
    name, file_info = _extract_file(change['new'])
    if not file_info:
        return
    img = Image.open(io.BytesIO(file_info['content'])).convert('RGB')
    img_resized = img.resize(IMG_SIZE)
    arr = np.array(img_resized) / 255.0
    x = np.expand_dims(arr, axis=0)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob >= THRESHOLD else CLASS_NAMES[0]
    with out_single:
        out_single.clear_output()
        print(f"Archivo: {name}")
        print(f"Prob. PNEUMONIA: {prob:.4f} | Clase: {label} | Umbral: {THRESHOLD}")
        display(img)
    upload_single.value = ()

upload_single.observe(on_upload_single, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen (.jpg/.png) para predecir:</b>'),
    upload_single,
    out_single
]))
